# NB16 — Clean up the repo structure (one-off)

`Shanmuk4622/ai-image-detection-dataset` accumulated a few stray items from earlier attempts:

- a top-level **`data/`** folder (old flat AI shards) — **superseded** by the new
  `real/ sd15/ sdxl/ flux_schnell/ kandinsky22/ pixart_sigma/ wuerstchen/` subset folders
  (each with `train/val/test`);
- a root **`config.json`** — a duplicate of `metadata/config.json`;
- **`_build_state.json`** — a build artifact that doesn't belong in a published dataset.

This notebook **verifies the real data is intact first**, then removes only the stray items.
Safe to run once. Kaggle: Internet ON, secret `HF_TOKEN` (write).

Resulting professional structure:
```
├── real/  sd15/  sdxl/  flux_schnell/  kandinsky22/  pixart_sigma/  wuerstchen/   (train/val/test)
├── captions/
├── metadata/   (config.json, manifest.parquet, splits.parquet, captions.parquet, validation_report.json)
├── ai_detect_common.py
├── README.md
└── .gitattributes
```


In [ ]:
import os
from huggingface_hub import HfApi
REPO="Shanmuk4622/ai-image-detection-dataset"; RT="dataset"
DELETE_BUILD_STATE=True   # remove _build_state.json (build is complete)

def token():
    try:
        from kaggle_secrets import UserSecretsClient
        t=UserSecretsClient().get_secret("HF_TOKEN")
        if t: return t.strip()
    except Exception: pass
    for k in ("HF_TOKEN","HUGGINGFACE_TOKEN","HUGGING_FACE_HUB_TOKEN"):
        if os.environ.get(k): return os.environ[k].strip()
    import getpass; return getpass.getpass("HF write token: ").strip()

api=HfApi(token=token())
files=api.list_repo_files(REPO,repo_type=RT)
SUBSETS=["real","sd15","sdxl","flux_schnell","kandinsky22","pixart_sigma","wuerstchen"]

# ---- SAFETY: every subset must have train/val/test parquet before we delete anything ----
print("verifying new subset folders are complete ...")
bad=[]
for s in SUBSETS:
    counts={sp:sum(1 for f in files if f.startswith(f"{s}/{sp}/") and f.endswith(".parquet"))
            for sp in ("train","val","test")}
    print(f"  {s:14s} train={counts['train']:4d} val={counts['val']:4d} test={counts['test']:4d}")
    if sum(counts.values())==0: bad.append(s)
assert not bad, f"ABORT: these subsets are empty, not safe to clean: {bad}"
print("all subsets present ✅")

In [ ]:
# ---- delete stray items ----
def has(path_prefix): return any(f==path_prefix or f.startswith(path_prefix+"/") for f in files)

if has("data"):
    api.delete_folder(path_in_repo="data",repo_id=REPO,repo_type=RT,commit_message="cleanup: remove stray data/ (superseded by subset folders)")
    print("deleted data/")
else: print("no data/ folder")

if "config.json" in files:
    api.delete_file("config.json",repo_id=REPO,repo_type=RT,commit_message="cleanup: drop root config.json (kept in metadata/)")
    print("deleted root config.json")

if DELETE_BUILD_STATE and "_build_state.json" in files:
    api.delete_file("_build_state.json",repo_id=REPO,repo_type=RT,commit_message="cleanup: remove build artifact")
    print("deleted _build_state.json")

# ---- show final structure ----
files=api.list_repo_files(REPO,repo_type=RT)
top=sorted({f.split('/')[0] for f in files})
print("\nfinal top-level entries:")
for t in top: print("  ", t + ("/" if any(x.startswith(t+"/") for x in files) else ""))
print("\nhttps://huggingface.co/datasets/%s/tree/main"%REPO)